# Agentic Synthetic Data Generation

In this tutorial, we build a lightweight agentic workflow for synthetic evaluation data generation.

The goal is to create a larger and cleaner dataset than we would get from a single generation call. We use three agents:
- a **generator** that creates candidate examples
- a **reviewer** that checks quality
- a **rewriter** that fixes examples that need revision

This is a controlled notebook workflow. We do not let agents run forever. Instead, we run a small `generate -> review -> revise -> collect` loop and inspect the results with `pandas`.

## 1. Setup

We use the OpenAI Agents SDK for lightweight agent orchestration.

Install it if needed:

In [ ]:
# !pip install openai-agents

In [3]:
import os
from typing import Any, Dict, List, Literal, Optional

import pandas as pd
from pydantic import BaseModel

from agents import Agent, Runner

pd.set_option("display.max_colwidth", None)

assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY before running this notebook."

## 2. Data schemas

Structured outputs keep every agent response easy to validate and convert into a DataFrame.

In [4]:
class SyntheticExample(BaseModel):
    category: str
    customer_question: str
    reference_answer: str
    question_length: Literal["short", "medium", "long"]
    persona: Optional[str] = None


class GeneratedBatch(BaseModel):
    examples: List[SyntheticExample]


class ReviewResult(BaseModel):
    example_id: int
    label: Literal["approved", "needs_revision", "rejected"]
    reasoning: str


class ReviewedBatch(BaseModel):
    reviews: List[ReviewResult]


class RevisedExample(BaseModel):
    example_id: int
    category: str
    customer_question: str
    reference_answer: str
    question_length: Literal["short", "medium", "long"]
    persona: Optional[str] = None


class RevisedBatch(BaseModel):
    examples: List[RevisedExample]

## 3. Generation config

The config defines what kind of dataset we want.

This keeps the workflow controlled.

In [5]:
generation_config = {
    "target_approved": 10,
    "batch_size": 6,
    "max_rounds": 3,
    "category_distribution": {
        "returns": 0.30,
        "shipping": 0.25,
        "payments": 0.15,
        "refunds": 0.15,
        "tracking": 0.10,
        "cancellations": 0.05,
    },
    "question_length_distribution": {
        "short": 0.45,
        "medium": 0.40,
        "long": 0.15,
    },
    "personas": [
        "angry customer",
        "confused first-time buyer",
        "impatient customer",
        "international customer",
        "budget-conscious customer",
        "anxious customer",
    ],
}

generation_config

{'target_approved': 10,
 'batch_size': 6,
 'max_rounds': 3,
 'category_distribution': {'returns': 0.3,
  'shipping': 0.25,
  'payments': 0.15,
  'refunds': 0.15,
  'tracking': 0.1,
  'cancellations': 0.05},
 'question_length_distribution': {'short': 0.45, 'medium': 0.4, 'long': 0.15},
 'personas': ['angry customer',
  'confused first-time buyer',
  'impatient customer',
  'international customer',
  'budget-conscious customer',
  'anxious customer']}

## 4. Agents

Each agent has a narrow role.

The generator creates candidates. The reviewer checks them. The rewriter fixes examples marked as `needs_revision`.

In [23]:
generator_agent = Agent(
    name="Synthetic data generator",
    instructions="""Generate synthetic evaluation examples for an e-commerce customer support assistant.

Create realistic customer questions and safe reference answers.

Follow the requested category distribution and question length distribution approximately.

Question length:
- short: 1-2 sentences
- medium: 3-4 sentences
- long: 5 or more sentences

Reference answers should be concise, safe, and generic. Do not invent unsupported store-specific policies.""",
    output_type=GeneratedBatch,
)


reviewer_agent = Agent(
    name="Synthetic data reviewer",
    model="gpt-5",
    instructions="""Review synthetic evaluation examples.

Approve an example if:
- the customer question is realistic
- the reference answer is safe and useful
- the category matches the question
- the question length matches the requested label: 1-2 sentences for "short"; 3-4 sentences for "medium"; 5 or more sentences for "long"
- the answer avoids unsupported store-specific claims

Use:
- approved: ready to use
- needs_revision: useful idea, but should be fixed
- rejected: low quality or not worth fixing""",
    output_type=ReviewedBatch,
)


rewriter_agent = Agent(
    name="Synthetic data rewriter",
    model="gpt-5",
    instructions="""Revise synthetic evaluation examples based on reviewer feedback.

Keep the same example_id.
Fix the question, reference answer, category, persona, or question_length when needed.
Return only revised examples.""",
    output_type=RevisedBatch,
)

## 5. Agent runners

These functions call each agent and convert structured outputs into DataFrames.

In [17]:
def config_to_prompt(config: Dict[str, Any]) -> str:
    return f"""Create {config["batch_size"]} candidate examples.

Category distribution:
{config["category_distribution"]}

Question length distribution:
{config["question_length_distribution"]}

Personas:
{config["personas"]}

Return structured examples only.
"""


async def generate_batch(config: Dict[str, Any]) -> pd.DataFrame:
    result = await Runner.run(
        generator_agent,
        input=config_to_prompt(config),
    )

    return pd.DataFrame([
        example.model_dump()
        for example in result.final_output.examples
    ])


async def review_batch(batch_df: pd.DataFrame) -> pd.DataFrame:
    payload = batch_df.reset_index(names="example_id").to_dict(orient="records")

    result = await Runner.run(
        reviewer_agent,
        input=f"Review these examples: {payload}",
    )

    return pd.DataFrame([
        {
            "example_id": review.example_id,
            "review_label": review.label,
            "review_reasoning": review.reasoning,
        }
        for review in result.final_output.reviews
    ])


async def rewrite_batch(batch_df: pd.DataFrame, review_df: pd.DataFrame) -> pd.DataFrame:
    needs_revision = review_df[review_df["review_label"] == "needs_revision"]

    if len(needs_revision) == 0:
        return pd.DataFrame(columns=[
            "example_id",
            "category",
            "customer_question",
            "reference_answer",
            "question_length",
            "persona",
        ])

    payload = {
        "examples": batch_df.reset_index(names="example_id").to_dict(orient="records"),
        "reviews": needs_revision.to_dict(orient="records"),
    }

    result = await Runner.run(
        rewriter_agent,
        input=f"Revise examples marked as needs_revision: {payload}",
    )

    return pd.DataFrame([
        example.model_dump()
        for example in result.final_output.examples
    ])

## 6. Run one generation round

First, we run one batch through the full workflow.

In [8]:
batch_df = await generate_batch(generation_config)

batch_df

,category,customer_question,reference_answer,question_length,persona
0,returns,I received the wrong size and I’m pretty annoyed. How do I return it and get the correct one?,"Sorry about that. Please contact support with your order number and a photo of the item, and they can guide you through the return or exchange steps.",short,angry customer
1,shipping,"I just placed my first order and I’m not sure what happens next. How long does shipping usually take, and when should I expect an email?","After an order is placed, you should usually receive an order confirmation email first, then a shipping update when it leaves the warehouse. Delivery time can vary by location and shipping method, so check your confirmation details for the best estimate.",medium,confused first-time buyer
2,payments,Why was my card charged twice? I only checked out once and I need this fixed quickly.,That can happen from a duplicate authorization or a payment processing issue. Please review your order history and contact support with the order number and the last four digits of the card so they can investigate.,short,impatient customer
3,refunds,"I’m outside the country and I sent something back last week. Will my refund be issued in my local currency or the original one, and how long might that take?","Refunds are usually processed back to the original payment method, and the currency handling depends on your bank or card provider. Processing time can vary, so it’s best to check with both the merchant and your payment provider if it takes longer than expected.",medium,international customer
4,tracking,"I’m trying to keep this under budget, so I need to know where my package is before it gets delayed. The tracking link says only that the label was created. Is that normal?","Yes, that can be normal for a short time after an order is packed. Tracking often updates once the carrier scans the package, so if it stays unchanged for several days, contact support for help.",medium,budget-conscious customer
5,returns,I’m worried I might miss the return window because I’m traveling and I can’t check my inbox often. Can I start a return now if I’m not ready to send it back today? I also want to make sure I don’t lose the item if I hold onto it for a few more days.,"In many cases you can start the return process before shipping the item back, but the exact timing depends on the seller’s policy. If you’re unsure, contact support as soon as possible with your order number so they can confirm the next step.",long,anxious customer


In [18]:
review_df = await review_batch(batch_df)

review_df

,example_id,review_label,review_reasoning
0,0,approved,"Realistic, correct category, short length, safe/general guidance."
1,1,needs_revision,Length mismatch (2 sentences vs. medium 3–4). Otherwise fine.
2,2,approved,"Realistic, correct category, short length, safe and useful answer."
3,3,needs_revision,Length mismatch (2 sentences vs. medium 3–4). Content is safe.
4,4,approved,"Realistic, correct category, medium length, safe answer."
5,5,needs_revision,Length mismatch (3 sentences vs. long 5+). Content is safe.


In [22]:
reviewed_batch_df = batch_df.reset_index(names="example_id").merge(
    review_df,
    on="example_id",
    how="left",
)

reviewed_batch_df

,example_id,category,customer_question,reference_answer,question_length,persona,review_label,review_reasoning
0,0,returns,I received the wrong size and I’m pretty annoyed. How do I return it and get the correct one?,"Sorry about that. Please contact support with your order number and a photo of the item, and they can guide you through the return or exchange steps.",short,angry customer,approved,"Realistic, correct category, short length, safe/general guidance."
1,1,shipping,"I just placed my first order and I’m not sure what happens next. How long does shipping usually take, and when should I expect an email?","After an order is placed, you should usually receive an order confirmation email first, then a shipping update when it leaves the warehouse. Delivery time can vary by location and shipping method, so check your confirmation details for the best estimate.",medium,confused first-time buyer,needs_revision,Length mismatch (2 sentences vs. medium 3–4). Otherwise fine.
2,2,payments,Why was my card charged twice? I only checked out once and I need this fixed quickly.,That can happen from a duplicate authorization or a payment processing issue. Please review your order history and contact support with the order number and the last four digits of the card so they can investigate.,short,impatient customer,approved,"Realistic, correct category, short length, safe and useful answer."
3,3,refunds,"I’m outside the country and I sent something back last week. Will my refund be issued in my local currency or the original one, and how long might that take?","Refunds are usually processed back to the original payment method, and the currency handling depends on your bank or card provider. Processing time can vary, so it’s best to check with both the merchant and your payment provider if it takes longer than expected.",medium,international customer,needs_revision,Length mismatch (2 sentences vs. medium 3–4). Content is safe.
4,4,tracking,"I’m trying to keep this under budget, so I need to know where my package is before it gets delayed. The tracking link says only that the label was created. Is that normal?","Yes, that can be normal for a short time after an order is packed. Tracking often updates once the carrier scans the package, so if it stays unchanged for several days, contact support for help.",medium,budget-conscious customer,approved,"Realistic, correct category, medium length, safe answer."
5,5,returns,I’m worried I might miss the return window because I’m traveling and I can’t check my inbox often. Can I start a return now if I’m not ready to send it back today? I also want to make sure I don’t lose the item if I hold onto it for a few more days.,"In many cases you can start the return process before shipping the item back, but the exact timing depends on the seller’s policy. If you’re unsure, contact support as soon as possible with your order number so they can confirm the next step.",long,anxious customer,needs_revision,Length mismatch (3 sentences vs. long 5+). Content is safe.


## 7. Revise examples

Only examples marked as `needs_revision` are sent to the rewriter.

In [24]:
revised_df = await rewrite_batch(batch_df, review_df)

revised_df

,example_id,category,customer_question,reference_answer,question_length,persona
0,1,shipping,"I just placed my first order and I’m not sure what happens next. How long does shipping usually take, and when should I expect an email?","After you place an order, you’ll first receive an order confirmation email, then a shipping notification as soon as it leaves the warehouse. Shipping times vary by location and method; most orders are packed within 1–2 business days, and the delivery estimate shown at checkout is your best guide. If you don’t see the emails, check your spam folder or update your email preferences.",medium,confused first-time buyer
1,3,refunds,"I’m outside the country and I sent something back last week. Will my refund be issued in my local currency or the original one, and how long might that take?","Refunds are returned to the original payment method, and any currency conversion is handled by your bank or card provider at their exchange rate. Once the return is received and processed, refunds typically post within 3–10 business days, though international transfers can take a bit longer. If it hasn’t appeared after that window, contact our support and your payment provider so they can trace it.",medium,international customer
2,5,returns,I’m worried I might miss the return window because I’m traveling and I can’t check my inbox often. Can I start a return now if I’m not ready to send it back today? I also want to make sure I don’t lose the item if I hold onto it for a few more days.,"In many cases you can open a return request now and ship the item later, but the exact window depends on the seller’s policy. Starting the request early will generate a return authorization and instructions without requiring immediate drop-off. You’ll usually have a set number of days after the label or RMA is issued to get the package to the carrier. If you’re traveling, you can still hold the item safely and ship once you’re back, as long as you stay within that window. To avoid missing it, contact support with your order number so they can confirm your deadline and help extend it if possible. They can also resend the label and note your account in case of travel delays.",long,anxious customer


## 8. Controlled loop

Now we run a small loop until we collect enough approved examples or reach `max_rounds`.

This keeps the agentic workflow bounded and reproducible.

In [25]:
async def run_agentic_generation_loop(config: Dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:
    approved_batches = []
    audit_log = []

    for round_id in range(1, config["max_rounds"] + 1):
        batch = await generate_batch(config)
        review = await review_batch(batch)

        reviewed = batch.reset_index(names="example_id").merge(
            review,
            on="example_id",
            how="left",
        )
        reviewed["round_id"] = round_id
        reviewed["source"] = "generated"

        approved_batches.append(
            reviewed[reviewed["review_label"] == "approved"].copy()
        )

        audit_log.append(reviewed)

        revised = await rewrite_batch(batch, review)

        if len(revised) > 0:
            revised_review = await review_batch(
                revised.drop(columns=["example_id"], errors="ignore")
            )

            revised_reviewed = (
                revised
                .drop(columns=["example_id"], errors="ignore")
                .reset_index(names="example_id")
                .merge(revised_review, on="example_id", how="left")
            )
            revised_reviewed["round_id"] = round_id
            revised_reviewed["source"] = "revised"

            approved_batches.append(
                revised_reviewed[revised_reviewed["review_label"] == "approved"].copy()
            )

            audit_log.append(revised_reviewed)

        approved_count = sum(len(batch) for batch in approved_batches)

        if approved_count >= config["target_approved"]:
            break

    final_df = pd.concat(approved_batches, ignore_index=True)
    audit_df = pd.concat(audit_log, ignore_index=True)

    return final_df.head(config["target_approved"]), audit_df


final_df, audit_df = await run_agentic_generation_loop(generation_config)

final_df

,example_id,category,customer_question,reference_answer,question_length,persona,review_label,review_reasoning,round_id,source
0,0,returns,I want to return the shoes I ordered because they don't fit. How do I start the return?,"I’m sorry the fit wasn’t right. Check the order details and return policy in your account or confirmation email, then follow the return steps provided there. If you can’t find it, contact support with your order number.",short,confused first-time buyer,approved,"Realistic, correct category, short length, safe generic guidance.",1,generated
1,1,shipping,My package hasn't moved in days and I need it soon. Can you tell me when it will arrive?,"I understand the wait is frustrating. Please review the latest tracking update for the most current status, and if it looks stalled, contact support with your order number so they can help investigate.",short,impatient customer,approved,"Realistic, correct category, short length, safe advice.",1,generated
2,5,cancellations,Can I cancel my order before it ships? I just realized I ordered the wrong size.,"If the order hasn’t been processed or shipped yet, cancellation may be possible. Please check your order status right away and contact support as soon as you can with your order number.",short,angry customer,approved,"Realistic, correct category, short length, safe generic steps.",1,generated
3,0,payments,"I'm trying to pay with my card, but the checkout keeps failing. Why is this happening?","That can happen for a few reasons, such as billing details not matching or a temporary bank issue. Please double-check your card info and try again, or use another payment method if available. If it still fails, contact your bank or support.",short,budget-conscious customer,approved,"Realistic payments question; concise, safe, and helpful answer; category fits; short length correct; no store-specific claims.",1,revised
4,1,refunds,"I returned my item last week, but I still don't see my refund. How long does it usually take?","Refund timing can vary depending on the payment method and when the return is processed. Please check your return status and payment account, and contact support with your order number if it’s taking longer than expected.",short,anxious customer,approved,"Realistic refunds question; useful, non-specific answer; correct category; short length; no unsupported claims.",1,revised
5,2,tracking,"I got a tracking number, but it doesn't show any updates yet. Is my order lost?","Not necessarily. Tracking can take time to update after a package ships, especially early on or with international delivery. Please check again later, and contact support if there’s still no movement after a reasonable time.",short,international customer,approved,"Realistic tracking question; safe, helpful guidance; correct category; short length; avoids store-specific promises.",1,revised
6,0,returns,I got the wrong size and I want to send it back. How do I start a return?,"I’m sorry about that. Check your order details for the return option or contact support with your order number, and they can help you start the return process.",short,angry customer,approved,"Realistic question; safe, useful answer; correct category; short length OK.",2,generated
7,1,shipping,I placed my order a few days ago and it still shows as processing. Can you tell me when it will ship? I’m a first-time buyer and I’m not sure what to expect.,"Processing times can vary. Please check your order confirmation or tracking page for the latest status, and contact support with your order number if it seems delayed.",medium,confused first-time buyer,approved,"Realistic; safe, useful answer; correct category; medium length OK.",2,generated
8,2,payments,My card was charged twice for the same order. What should I do?,That can happen from a duplicate authorization or payment error. Please contact support with your order number and payment details so they can review it.,short,impatient customer,approved,"Realistic; safe, useful a

## 9. Inspect the audit log

The audit log is the simplest form of observability.

It shows what was generated, what was rejected, and why.

In [26]:
audit_df[[
    "round_id",
    "source",
    "category",
    "question_length",
    "customer_question",
    "review_label",
    "review_reasoning",
]]

,round_id,source,category,question_length,customer_question,review_label,review_reasoning
0,1,generated,returns,short,I want to return the shoes I ordered because they don't fit. How do I start the return?,approved,"Realistic, correct category, short length, safe generic guidance."
1,1,generated,shipping,short,My package hasn't moved in days and I need it soon. Can you tell me when it will arrive?,approved,"Realistic, correct category, short length, safe advice."
2,1,generated,payments,medium,"I'm trying to pay with my card, but the checkout keeps failing. Why is this happening?",needs_revision,Length mismatch: question is short but labeled medium.
3,1,generated,refunds,medium,"I returned my item last week, but I still don't see my refund. How long does it usually take?",needs_revision,Length mismatch: question is short but labeled medium.
4,1,generated,tracking,medium,"I got a tracking number, but it doesn't show any updates yet. Is my order lost?",needs_revision,Length mismatch: question is short but labeled medium.
5,1,generated,cancellations,short,Can I cancel my order before it ships? I just realized I ordered the wrong size.,approved,"Realistic, correct category, short length, safe generic steps."
6,1,revised,payments,short,"I'm trying to pay with my card, but the checkout keeps failing. Why is this happening?",approved,"Realistic payments question; concise, safe, and helpful answer; category fits; short length correct; no store-specific claims."
7,1,revised,refunds,short,"I returned my item last week, but I still don't see my refund. How long does it usually take?",approved,"Realistic refunds question; useful, non-specific answer; correct category; short length; no unsupported claims."
8,1,revised,tracking,short,"I got a tracking number, but it doesn't show any updates yet. Is my order lost?",approved,"Realistic tracking question; safe, helpful guidance; correct category; short length; avoids store-specific promises."
9,2,generated,returns,short,I got the wrong size and I want to send it back. How do I start a return?,approved,"Realistic question; safe, useful answer; correct category; short length OK."


## 10. Quality summary

We can summarize acceptance rate and rejection reasons.

In [27]:
label_counts = audit_df["review_label"].value_counts()
acceptance_rate = (audit_df["review_label"] == "approved").mean()

print(f"Acceptance rate: {acceptance_rate:.1%}")
display(label_counts)

Acceptance rate: 66.7%


review_label
approved          12
needs_revision     6
Name: count, dtype: int64

In [28]:
category_coverage = final_df["category"].value_counts()
length_coverage = final_df["question_length"].value_counts()

display(category_coverage)
display(length_coverage)

category
returns          2
shipping         2
payments         2
refunds          2
cancellations    1
tracking         1
Name: count, dtype: int64

question_length
short     8
medium    2
Name: count, dtype: int64

## 11. Export the dataset

The final dataset contains approved examples only.

In a real workflow, this file should still go through human review before becoming a trusted evaluation dataset.

In [29]:
output_path = "agentic_synthetic_evaluation_dataset.csv"

final_df.to_csv(output_path, index=False)

output_path

'agentic_synthetic_evaluation_dataset.csv'

## 12. Takeaways

Agentic synthetic data generation turns dataset creation into an iterative workflow.

The generator creates candidates, the reviewer applies quality criteria, and the rewriter improves examples that are worth fixing.

This can scale dataset creation, but it should not remove human review. The main benefit is a more systematic and observable process.